In [ ]:
#| default_exp send_alert


# Step 7: Slack notifications

![Step 7 diagram](https://raw.githubusercontent.com/jaewilson07/bird-watcher/main/docs/diagrams/07-step.png)

**Goal:** every time we save a sighting, post a message to Slack.

We use a Slack **incoming webhook** — just a URL that accepts JSON. If you don't have a webhook set yet, the code falls back to a *dry-run* that prints the message instead.

Two functions live in `bird_watcher/send_alert.py`:

- `build_sighting_alert` — turn a saved `Sighting` into Slack Block Kit JSON
- `send_alert_to_slack` — POST it (or dry-run)

## Step 7.0 — Setup

Don't have a webhook yet? The app manifest lives at [`setup/slack-manifest.yml`](../setup/slack-manifest.yml) — go to [api.slack.com/apps](https://api.slack.com/apps) → **Create New App** → **From a manifest**, paste it in, install to your workspace, pick a channel, and copy the webhook URL it gives you.

The Slack webhook URL is a **secret** — anyone who has it can post to your channel — so it lives in `.env`, never in code.

If `SLACK_WEBHOOK` is empty, everything below runs in *dry-run* mode and just prints the message instead of sending it.

In [ ]:
from dotenv import load_dotenv
from pathlib import Path
import os

assert load_dotenv(), "Failed to load .env file"

SLACK_WEBHOOK = os.environ.get("SLACK_WEBHOOK", "")

DATA_FOLDER = Path("../data")
DB_PATH = DATA_FOLDER / "birds.db"
SNAPSHOT_FOLDER = DATA_FOLDER / "snapshots"
SNAPSHOT_FOLDER.mkdir(parents=True, exist_ok=True)

print(f"Slack webhook set: {bool(SLACK_WEBHOOK)}")


## Step 7.1 — Slack messages use Block Kit

Slack Block Kit is JSON that describes the message layout. A header + a section with fields is enough for a sighting alert.

In [ ]:
payload = {
    "blocks": [
        {
            "type": "header",
            "text": {"type": "plain_text", "text": ":bird: New sighting — Northern Cardinal"},
        },
        {
            "type": "section",
            "fields": [
                {"type": "mrkdwn", "text": "*Species*\nNorthern Cardinal"},
                {"type": "mrkdwn", "text": "*Confidence*\n92%"},
            ],
        },
    ]
}
import json
print(json.dumps(payload, indent=2))

## Step 7.2 — Wrap the payload builder as `build_sighting_alert`

In [ ]:
#| export
from bird_watcher.persistence import Sighting

def build_sighting_alert(sighting: Sighting) -> dict:
    """Build a Slack Block Kit message payload for one sighting.

    Args:
        sighting: the saved Sighting to announce.

    Returns:
        A dict ready to be POSTed as JSON to a Slack webhook.
    """
    confidence_pct = int(sighting.confidence * 100)

    header_text = f":bird: New sighting — {sighting.species}"
    if sighting.id is not None:
        header_text += f" (#{sighting.id})"

    return {
        "blocks": [
            {
                "type": "header",
                "text": {"type": "plain_text", "text": header_text},
            },
            {
                "type": "section",
                "fields": [
                    {"type": "mrkdwn", "text": f"*Species*\n{sighting.species}"},
                    {"type": "mrkdwn", "text": f"*Confidence*\n{confidence_pct}%"},
                    {"type": "mrkdwn", "text": f"*Seen at*\n{sighting.timestamp:%I:%M %p}"},
                ],
            },
        ]
    }


## Step 7.3 — Add `send_alert_to_slack`

In [ ]:
#| export
import json

import requests


def send_alert_to_slack(
    payload: dict,
    webhook_url: str | None = None,
    debug_prn: bool = False,
) -> bool:
    """POST a message payload to Slack. Prints locally if no webhook.

    Args:
        payload: the dict returned by `build_sighting_alert`.
        webhook_url: the Slack incoming webhook URL. If empty, this runs in
            dry-run mode and prints the payload instead.
        debug_prn: if True, print whether we sent or just previewed.

    Returns:
        True if the message was sent (or dry-run previewed), False if failed.
    """
    url = webhook_url or ""
    if not url:
        if debug_prn:
            print("[dry-run] No SLACK_WEBHOOK set. Would have sent:")
            print(json.dumps(payload, indent=2))
        return True

    try:
        response = requests.post(
            url,
            data=json.dumps(payload),
            headers={"Content-Type": "application/json"},
            timeout=10,
        )
        response.raise_for_status()
        if debug_prn:
            print(f"Sent to Slack (status {response.status_code})")
        return True
    except requests.RequestException as exc:
        if debug_prn:
            print(f"Slack send failed: {exc}")
        return False


## Acceptance criterion — wire recording to alerting

Time to deliver on this lesson's goal: *every time we SAVE a sighting, post to Slack.*

The key is what `record_sighting` returns: a response when it saved, `None` when the bird was inside its cooldown window. That `None` is the signal **not** to buzz your phone — the cardinal sitting at the feeder for ten minutes is one visit, not ten.

Notice *where* the wiring lives: out here at the call site. The storage layer doesn't know Slack exists, and Slack doesn't know SQLite exists — same separation as the web app in lesson 9. Each piece keeps one job.

In [ ]:
from bird_watcher.persistence import Sighting, SqliteSightingStorage
from bird_watcher.send_alert import build_sighting_alert, send_alert_to_slack

storage = SqliteSightingStorage(db_path=DB_PATH, cooldown_minutes=0)

jpg_files = sorted(SNAPSHOT_FOLDER.glob("*.jpg"))
image_bytes = jpg_files[0].read_bytes() if jpg_files else b"pretend this is a jpg"

sighting = Sighting(species="Northern Cardinal", confidence=0.92, image_bytes=image_bytes)

response = storage.record_sighting(sighting, debug_prn=True)

if response:  # None = seen within the cooldown — a repeat visit shouldn't buzz your phone
    sighting.id = response.last_row_id  # the database just assigned the id (lesson 6!)

    payload = build_sighting_alert(sighting)
    sent = send_alert_to_slack(payload, webhook_url=SLACK_WEBHOOK, debug_prn=True)
    assert sent, "send_alert_to_slack should return True (dry-run if no webhook)"

print("✅ Notification flow works: record → (only if new) → alert")

## What's next

**Step 8:** open [08-web-hello.ipynb](08-web-hello.ipynb) — we'll start a tiny web app with FastAPI.